In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [5]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_23_9_7,0.999422,0.774091,0.998031,0.993449,0.997913,0.003864,1.510657,0.009555,0.011516,0.010536,0.109545,0.062158,1.000322,0.064804,145.112312,226.776993,"Hidden Size=[7, 5], regularizer=0.3, learning_..."
1,model_23_9_8,0.999421,0.774159,0.998094,0.992923,0.997852,0.003869,1.510197,0.009248,0.012441,0.010844,0.106211,0.062200,1.000323,0.064848,145.109603,226.774283,"Hidden Size=[7, 5], regularizer=0.3, learning_..."
2,model_23_9_6,0.999420,0.774010,0.997956,0.994010,0.997975,0.003879,1.511196,0.009918,0.010530,0.010224,0.113232,0.062285,1.000324,0.064937,145.104133,226.768813,"Hidden Size=[7, 5], regularizer=0.3, learning_..."
3,model_23_9_9,0.999418,0.774218,0.998148,0.992433,0.997792,0.003889,1.509804,0.008987,0.013303,0.011145,0.103199,0.062364,1.000325,0.065019,145.099079,226.763759,"Hidden Size=[7, 5], regularizer=0.3, learning_..."
4,model_23_9_10,0.999414,0.774268,0.998194,0.991977,0.997735,0.003920,1.509468,0.008765,0.014103,0.011434,0.100483,0.062613,1.000327,0.065279,145.083099,226.747779,"Hidden Size=[7, 5], regularizer=0.3, learning_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
303,model_25_9_20,0.981212,0.794465,0.999974,0.992022,0.993878,0.125634,1.374415,0.000042,0.057504,0.028773,0.210348,0.354448,1.008671,0.369538,156.148770,248.783333,"Hidden Size=[7, 6], regularizer=0.5, learning_..."
304,model_25_9_16,0.981212,0.794465,0.999974,0.992022,0.993878,0.125634,1.374415,0.000042,0.057504,0.028773,0.210348,0.354448,1.008671,0.369538,156.148770,248.783333,"Hidden Size=[7, 6], regularizer=0.5, learning_..."
305,model_25_9_15,0.981212,0.794465,0.999974,0.992022,0.993878,0.125634,1.374415,0.000042,0.057504,0.028773,0.210348,0.354448,1.008671,0.369538,156.148770,248.783333,"Hidden Size=[7, 6], regularizer=0.5, learning_..."
306,model_25_9_14,0.981212,0.794465,0.999974,0.992022,0.993878,0.125634,1.374415,0.000042,0.057504,0.028773,0.210348,0.354448,1.008671,0.369538,156.148770,248.783333,"Hidden Size=[7, 6], regularizer=0.5, learning_..."
